# Enterprise Intelligence & Customer Retention Platform

Core pipeline: **Ingestion → Validation → Cleaning → EDA → Cohorts → RFM → Historical Snapshot → Churn Model → Risk Segmentation → Executive Outputs**.

Set `DATA_PATH` to the supplied CSV/XLSX dataset before running.


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")
DATA_PATH = "transactions.csv"
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
HIGH_RISK_THRESHOLD = 0.70
MEDIUM_RISK_THRESHOLD = 0.40


## 1. Data Ingestion and Column Mapping

In [ ]:
def load_data(path):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(f"Dataset not found: {path.resolve()}")
    if path.suffix.lower()==".csv": return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx",".xls"}: return pd.read_excel(path)
    raise ValueError("Use CSV or Excel.")
raw_df=load_data(DATA_PATH)
df=raw_df.copy(deep=True)
def norm(x): return re.sub(r"[^A-Za-z0-9]+","_",str(x).strip()).strip("_").lower()
aliases={"order_id":["order_id","orderid","invoice_id","invoiceid"],"customer_id":["customer_id","customerid","customer"],"order_date":["order_date","orderdate","date"],"product":["product","product_name","sku","item"],"category":["category","product_category"],"region":["region","city","branch"],"quantity":["quantity","qty"],"revenue":["revenue","sales","amount","total_sales"],"profit":["profit","gross_profit"]}
lookup={v:k for k,vals in aliases.items() for v in vals}
df=df.rename(columns={c:lookup[norm(c)] for c in df.columns if norm(c) in lookup})
print(df.shape); display(df.head())


## 2. Data Quality Audit

In [ ]:
def audit(data):
    issues=[]
    for col in data.columns:
        n=int(data[col].isna().sum())
        if n: issues.append({"column":col,"issue":"missing_values","count":n,"severity":"WARNING"})
    d=int(data.duplicated().sum())
    if d: issues.append({"column":"__row__","issue":"duplicate_rows","count":d,"severity":"WARNING"})
    if "order_date" in data:
        bad=int(pd.to_datetime(data["order_date"],errors="coerce",dayfirst=True).isna().sum())
        if bad: issues.append({"column":"order_date","issue":"invalid_dates","count":bad,"severity":"ERROR"})
    if "quantity" in data:
        q=pd.to_numeric(data["quantity"],errors="coerce"); n=int((q<0).sum())
        if n: issues.append({"column":"quantity","issue":"negative_quantity","count":n,"severity":"WARNING"})
    if "product" in data:
        n=int(data["product"].astype("string").str.strip().eq("").sum())
        if n: issues.append({"column":"product","issue":"blank_product","count":n,"severity":"WARNING"})
    return pd.DataFrame(issues)
quality_report=audit(df); display(quality_report if not quality_report.empty else pd.DataFrame([{"status":"No configured issues detected"}]))


## 3. Cleaning and Data Dictionary

In [ ]:
def clean_currency(x):
    if pd.isna(x): return np.nan
    s=re.sub(r"[^0-9.\-]","",str(x).replace(",","").replace("₹","").replace("INR",""))
    try: return float(s) if s else np.nan
    except: return np.nan
clean_df=df.copy(deep=True)
for c in ["product","category","region"]:
    if c in clean_df: clean_df[c]=clean_df[c].astype("string").str.strip().str.replace(r"\s+"," ",regex=True)
if "order_date" in clean_df: clean_df["order_date"]=pd.to_datetime(clean_df["order_date"],errors="coerce",dayfirst=True)
for c in ["quantity","revenue","profit"]:
    if c in clean_df: clean_df[c]=clean_df[c].map(clean_currency) if c in ["revenue","profit"] else pd.to_numeric(clean_df[c],errors="coerce")
clean_df=clean_df.drop_duplicates().reset_index(drop=True)
dictionary=pd.DataFrame([{"column":c,"dtype":str(clean_df[c].dtype),"missing_count":int(clean_df[c].isna().sum()),"unique_count":int(clean_df[c].nunique(dropna=True))} for c in clean_df.columns])
dictionary.to_csv(OUTPUT_DIR/"data_dictionary.csv",index=False)
display(dictionary)


## 4. Business KPIs and EDA

In [ ]:
kpis={}
if "revenue" in clean_df: kpis.update({"Total Revenue":clean_df.revenue.sum(),"Average Order Value":clean_df.revenue.mean()})
if "profit" in clean_df: kpis["Total Profit"]=clean_df.profit.sum()
if "order_id" in clean_df: kpis["Orders"]=clean_df.order_id.nunique()
if "customer_id" in clean_df: kpis["Customers"]=clean_df.customer_id.nunique()
for k,v in kpis.items(): print(f"{k}: {v:,.2f}" if isinstance(v,(float,np.floating)) else f"{k}: {v:,}")
if {"order_date","revenue"}.issubset(clean_df.columns):
    monthly=clean_df.assign(month=clean_df.order_date.dt.to_period("M").astype(str)).groupby("month").revenue.sum()
    plt.figure(figsize=(10,5)); plt.plot(monthly.index,monthly.values,marker="o"); plt.title("Monthly Revenue"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if {"category","revenue"}.issubset(clean_df.columns):
    cat=clean_df.groupby("category").revenue.sum().sort_values(ascending=False)
    plt.figure(figsize=(10,5)); cat.plot(kind="bar"); plt.title("Revenue by Category"); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()
if {"region","revenue"}.issubset(clean_df.columns):
    reg=clean_df.groupby("region").revenue.sum().sort_values()
    plt.figure(figsize=(10,5)); reg.plot(kind="barh"); plt.title("Revenue by Region"); plt.tight_layout(); plt.show()


## 5. Customer Cohorts and RFM Historical Snapshot

In [ ]:
required={"customer_id","order_id","order_date","revenue"}
if required.issubset(clean_df.columns):
    counts=clean_df.groupby("customer_id").order_id.nunique()
    cohorts=pd.DataFrame({"customer_id":counts.index,"order_count":counts.values})
    cohorts["cohort"]=np.where(cohorts.order_count==1,"New Customer","Repeat Customer")
    display(cohorts.cohort.value_counts().rename_axis("cohort").reset_index(name="customers"))
    valid=clean_df.order_date.dropna(); observation_end=valid.max()-pd.Timedelta(days=30); future_end=valid.max()
    observation=clean_df[clean_df.order_date<=observation_end].copy(); future=clean_df[(clean_df.order_date>observation_end)&(clean_df.order_date<=future_end)].copy()
    rfm=observation.groupby("customer_id").agg(last_purchase=("order_date","max"),frequency=("order_id","nunique"),monetary=("revenue","sum")).reset_index()
    rfm["recency"]=(observation_end-rfm.last_purchase).dt.days; rfm["aov"]=rfm.monetary/rfm.frequency.replace(0,np.nan)
    future_customers=set(future.customer_id.dropna()); rfm["churn"]=(~rfm.customer_id.isin(future_customers)).astype(int)
    print("Observation end:",observation_end.date(),"| Future end:",future_end.date()); display(rfm.head())
else: print("RFM requires customer_id, order_id, order_date and revenue.")


## 6. Leakage-Safe Logistic Regression Churn Model

In [ ]:
features=["recency","frequency","monetary","aov"]
if "rfm" in globals() and not rfm.empty:
    model_df=rfm.dropna(subset=features+["churn"]).copy(); X=model_df[features]; y=model_df.churn
    if len(model_df)>=20 and y.nunique()==2:
        X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y)
        model=Pipeline([("scaler",StandardScaler()),("classifier",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))])
        model.fit(X_train,y_train); pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
        metrics={"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred,zero_division=0),"Recall":recall_score(y_test,pred,zero_division=0),"F1":f1_score(y_test,pred,zero_division=0)}
        if y_test.nunique()==2: metrics["ROC-AUC"]=roc_auc_score(y_test,prob)
        for k,v in metrics.items(): print(f"{k}: {v:.4f}")
        print(classification_report(y_test,pred,zero_division=0)); print("Confusion matrix:\n",confusion_matrix(y_test,pred))
    else: print("Insufficient data or classes for churn training.")


## 7. Risk Scoring and Exports

In [ ]:
if "model" in globals():
    scored=rfm.dropna(subset=features).copy(); scored["churn_probability"]=model.predict_proba(scored[features])[:,1]
    scored["risk_tier"]=np.select([scored.churn_probability>=HIGH_RISK_THRESHOLD,scored.churn_probability>=MEDIUM_RISK_THRESHOLD],["High Risk","Medium Risk"],default="Low Risk")
    risk=scored[["customer_id","recency","frequency","monetary","aov","churn_probability","risk_tier"]].sort_values("churn_probability",ascending=False)
    display(risk.head(20)); risk.to_csv(OUTPUT_DIR/"customer_risk_scores.csv",index=False)
clean_df.to_csv(OUTPUT_DIR/"cleaned_transactions.csv",index=False)
quality_report.to_csv(OUTPUT_DIR/"data_quality_report.csv",index=False)
if "rfm" in globals(): rfm.to_csv(OUTPUT_DIR/"customer_rfm.csv",index=False)
print("Outputs:", [p.name for p in sorted(OUTPUT_DIR.iterdir())])


## 8. Final Validation Checklist

- Raw data remains unchanged.
- Data-quality findings are reported.
- Cleaning is reproducible.
- EDA is dynamic.
- RFM uses only the observation window.
- Churn labels use the future window.
- No future feature is passed to the model.
- Model metrics are calculated from the current run.
- Risk thresholds are configurable.
- Output files are generated.
